# Statmodels Model

In [ ]:
# imports
import os
import json
import pickle
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from Encoder import encode, decode

### Dataset

In [ ]:
# data read
data = pd.read_csv('/content/processed_car_sales_data_cleaning.csv')
data.head()

In [ ]:
# Columns division by type

# numerical columns
numerical = data[['Engine size', 'Year of manufacture', 'Mileage', 'Price']]

# catigorical columns
catigorical = data[['Manufacturer', 'Model','Fuel type']]

# change data types into categorey
data[catigorical.columns] = data[catigorical.columns].astype('category')

# one hot encoding for categorical data
data = encode(data)

In [ ]:
# Set plot styles
plt.style.use('default')
sns.set_palette("deep")

In [ ]:
# Feature selection
X = data.drop("Price", axis=1)  # Input features
y = data["Price"]  # Target: Price

In [ ]:
# Features Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
# Reset indices to avoid alignment issues
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Add intercept column
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)

### Model

In [ ]:
# Model fit
ols_model = sm.OLS(y_train, X_train).fit()
print(ols_model.summary())

# Predictions
y_train_pred = ols_model.predict(X_train)
y_test_pred = ols_model.predict(X_test)

##### Model Evaluation

In [ ]:
# Model Evaluation
print("\n--- Train Performance ---")
print("Train R²:", r2_score(y_train, y_train_pred))
print("Train RMSE:", np.sqrt(mean_squared_error(y_train, y_train_pred)))

print("\n--- Test Performance ---")
print("Test R²:", r2_score(y_test, y_test_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_test_pred)))

In [ ]:
# Learning Curve
lr = LinearRegression()
train_sizes, train_scores, test_scores = learning_curve(
    lr, X_scaled, y, cv=5, scoring="r2", train_sizes=np.linspace(0.1, 1.0, 5)
)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, label="Train R²")
plt.plot(train_sizes, test_mean, label="Test R²")
plt.xlabel("Training Samples")
plt.ylabel("R² Score")
plt.title("Learning Curve (OLS approximation)")
plt.legend()
plt.show()

In [ ]:
# Model save
joblib.dump(ols_model, "statsmodels.joblib", compress = 3)
joblib.dump(scaler, os.path.join("scaler.pkl"))

In [ ]:
# Save metadata
features_list = data.drop("Price", axis=1).columns.tolist()

model_metadata = {
    'features': features_list,
    'target': 'Price',
    'model_type': 'OLS',
    'scaler': 'StandardScaler'
}

with open("model_metadata.json", "w") as f:
    json.dump(model_metadata, f)

### Prediction

In [ ]:
# Load the model
loaded_price_model = joblib.load('/content/st_model.joblib') 

In [ ]:
# Create example data for prediction
example_data = pd.DataFrame([
    [1.8, 2015, 80000, 'Toyota', 'Yaris', 'Petrol'],
    [3.0, 2020, 20000, 'Porsche', '911', 'Petrol'],
    [2.0, 2018, 60000, 'VW', 'Golf', 'Hybrid'],
    [1.4, 2012, 120000, 'Ford', 'Fiesta', 'Petrol'],  # Ford Fiesta, Petrol
])

#encode
example_data = encode(example_data)

# Align columns with training set
example_data = example_data.reindex(
    columns=loaded_price_model.model.exog_names, fill_value=0
)

In [ ]:
# Make predictions
price_predictions = loaded_price_model.predict(example_data)

# Decode categorical columns
decoded_rows = decode(example_data)

for i in example_data.iterrows():
    decoded_rows.append({
        "Predicted Price": price_predictions[i]
    })

results = pd.DataFrame(decoded_rows)

# Display predictions
print("\nPredictions for new cars:")
print(results)